# Word-bank approach: cost-aware reasoning comparison

**Implementation revision: word-bank-search-3.** This notebook compares tree search
methods with Standard CoT. Keep its results separate from earlier 16-method pilots.

**Model: Qwen 2.5:7B through Ollama. Upload this notebook alone to Kaggle.**
Enable **Internet** and a **GPU accelerator**, then Run All. No repository upload is needed.
The default pilot runs 5 development questions × 9 methods × 1 seed = **45 solver runs**.
Search methods can make many calls; this is not a 45-call experiment.

This is a separate experiment. It does not replace the original notebook or its results.
All code is ordinary readable Python, with no escaped source blobs or generated-code `exec`.
First-time setup downloads Ollama, Qwen, and the public datasets. The default pilot is a
smoke test, not sufficient evidence for a research conclusion.

## What does the word bank mean here?

We select 200 GSM8K **training** examples, excluding development questions, all exact
GSM8K test matches, and all exact SVAMP matches. Their official worked solutions supply
positive reference steps. A frozen TF-IDF vectorizer represents words and two-word phrases.
For a new question, retrieve its three closest training questions locally; a candidate step's
bank score is its maximum cosine similarity to the reference steps for those questions.
The bank is never inserted wholesale into the LLM prompt. No API call is needed to score it.

This is a **reference-derived phrase bank**, not a classifier trained on human preferences.
There are no negative step labels. Similar wording is not proof of useful progress or correctness.
The exported global phrase table helps inspection; ranking uses the context-specific subset.
Numbers are removed from lexical features. Full training solutions are used locally to
derive phrase scores; they are not placed in LLM prompts. Evaluation answers never enter solver prompts or ranking.
Exact-match separation cannot rule out paraphrases or the model's pretraining exposure.

## Implemented comparisons

| Method | What changes | Main cost or limitation |
|---|---|---|
| Standard CoT | One full solution, up to 4096 output tokens | Strong baseline; no terminal reviewer |
| Tree-of-Thoughts (LLM-Judge) | Beam search, LLM scores candidates | Judge calls count in total tokens |
| Cost-Aware Priority Search | TF-IDF + predicted next-call token cost | Predicted cost is approximate |
| Quality-only ablation | Same search with cost weight zero | Isolates the token cost term |
| Thought of Search adaptation | Generate restricted search code, run local BFS, audit semantics | Extra audit calls count toward cost |
| Local Search: No Lexical Features | Arithmetic gate, novelty proxy, estimated next-call cost | Control for new search implementation |
| Local Search: Question TF-IDF | Add candidate–question similarity | Tests ordinary lexical relevance |
| Local Search: TF-IDF + Word Bank | Add similarity to relevant training reference steps | Main word-bank treatment |
| Local Search: Word Bank + Compact State | Send labels/equations instead of full step objects | May discard useful explanations |

All four new local-search variants share candidate generation, beam/depth limits and a
counted terminal LLM review. The last variant changes history serialization only.
Rejected terminal candidates receive one counted independent appeal and bounded recovery
feedback under the same budgets. Identical rejected terminal paths use a cached decision.
Branches with no usable candidates also receive bounded repair feedback. A verified partial
calculation is never automatically promoted to a final answer; completion and review are required.

## Search heuristic and fair interpretation

Invalid arithmetic is rejected. For surviving candidates:

`utility = 0.2 + 0.5*novel_equation - 0.1*repetition + 0.15*question_tfidf + 0.15*bank_score`

`priority = utility / estimated_next_call_tokens`

The appropriate lexical terms are disabled for ablations. Novel arithmetic is a weak
progress proxy: it does not establish that a required subgoal was solved. Token cost uses
estimated next-prompt size plus observed mean generation length, not estimated total cost
to a correct answer. The fixed weights are initial experimental settings, not learned optima.
Tune only on development data and freeze choices before evaluation.

Ranking generated candidates saves only potential **future** expansions. It does not refund
their generation tokens. All generation, repairs, judging, reviews and failed attempts are logged.
The shared per-problem budget uses approximate input-token preflight and actual returned usage.
It is **not a strict billing cap**; overshoots and requests with missing usage are reported.
If any request has unavailable/invalid usage, the affected method's aggregate cost is left
undefined rather than presenting partial usage as token savings. Observed usage is retained separately.
Existing baselines use this notebook's shared limits, so do not pool with older unrestricted runs.

## Primary references

- [Thought of Search](https://arxiv.org/abs/2404.11833): generated search components; our adaptation
  has no human semantic validation or domain-level cost amortization.
- [ChemReasoner](https://proceedings.mlr.press/v235/sprueill24a.html): professor-provided motivation
  for search with external feedback; not reproduced as an arithmetic baseline here.
- [LLMLingua](https://arxiv.org/abs/2310.05736): compression motivation; compact state here is deterministic,
  not the LLMLingua algorithm.
- [GSM8K](https://github.com/openai/grade-school-math) and
  [SVAMP](https://github.com/arkilpatel/SVAMP): checksum-verified official source snapshots.


## 1. Kaggle setup


In [ ]:
# Kaggle: Internet ON; GPU selected. Run this once per new session.
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from pathlib import Path

WORK_DIR = Path('/kaggle/working')
if not WORK_DIR.is_dir():
    raise RuntimeError('This setup cell is intended for Kaggle /kaggle/working.')
os.chdir(WORK_DIR)
subprocess.run(['nvidia-smi'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'ollama', 'scikit-learn', 'pandas', 'matplotlib'], check=True)
if shutil.which('ollama') is None:
    subprocess.run(['apt-get', 'update', '-y'], check=True)
    subprocess.run(['apt-get', 'install', '-y', 'zstd', 'pciutils'], check=True)
    installer = WORK_DIR / 'install_ollama_word_bank.sh'
    urllib.request.urlretrieve('https://ollama.com/install.sh', installer)
    subprocess.run(['sh', str(installer)], check=True)

def server_ready():
    try:
        with urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2):
            return True
    except Exception:
        return False

if not server_ready():
    server_log = open(WORK_DIR / 'word_bank_ollama.log', 'a')
    server_process = subprocess.Popen(['ollama', 'serve'], stdout=server_log, stderr=server_log)
    for attempt in range(60):
        if server_ready():
            break
        time.sleep(1)
    else:
        raise RuntimeError('Ollama failed to start; inspect word_bank_ollama.log.')
subprocess.run(['ollama', 'pull', 'qwen2.5:7b'], check=True)
print('Setup ready. Model download is not benchmark inference cost.')


In [ ]:
# Actual Ollama usage; no application response cache and no silent API retries.
Model = 'qwen2.5:7b'

def ollama_backend(prompt, schema, temperature, seed, max_tokens):
    import ollama
    client = ollama.Client(host='http://127.0.0.1:11434', timeout=180)
    kwargs = dict(
        model=Model, messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature, 'seed': seed,
                 'num_predict': max_tokens, 'num_ctx': 8192},
    )
    if schema is not None:
        kwargs['format'] = schema
    response = client.chat(**kwargs)
    return {'text': response['message']['content'],
            'prompt_tokens': response.get('prompt_eval_count'),
            'completion_tokens': response.get('eval_count'),
            'done_reason': response.get('done_reason', 'unknown')}


## Embedded source: benchmark_engine


In [ ]:
"""Shared, traceable benchmark implementation; also embedded in the Kaggle notebook.

No solver receives the reference answer until scoring. Unfinished branches never
become answers. Arithmetic checks verify equalities, not word-problem semantics.
"""
import ast
import heapq
import itertools
import json
import math
import operator
import re
import time
from dataclasses import dataclass

VERSION = '4'
METHODS = ('Standard CoT (1-Call)', 'Step-wise Greedy CoT',
           'Tree-of-Thoughts (LLM-Judge)', 'Cost-Aware Priority Search',
           'Ablation: Quality-Only (w3=0)', 'Thought of Search (Arithmetic Adaptation)')

STEP_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {
        'step': {'type': 'string'}, 'equation': {'type': 'string'},
        'done': {'type': 'boolean'}, 'answer': {'type': ['number', 'null']},
        'target': {'type': 'string'},
        'expression': {'type': 'string'}, 'result': {'type': ['number', 'null']},
    }, 'required': ['step', 'equation', 'done', 'answer', 'target'],
}
REVIEW_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {'valid': {'type': 'boolean'}, 'answer': {'type': 'number'},
                   'reason': {'type': 'string'}},
    'required': ['valid', 'answer', 'reason'],
}
JUDGE_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {'score': {'type': 'number', 'minimum': 0, 'maximum': 1}},
    'required': ['score'],
}




In [ ]:
@dataclass(frozen=True)
class Config:
    seed: int = 11
    max_depth: int = 10
    beam: int = 3
    candidates: int = 2
    max_expansions: int = 40
    max_calls: int = 240
    retries: int = 2
    step_tokens: int = 512
    final_tokens: int = 1536
    max_tokens: int = 4096
    w1: float = 0.5
    audit_tos: bool = False
    review_appeal: bool = False

    def __post_init__(self):
        for key in ('max_depth', 'beam', 'candidates', 'max_expansions', 'max_calls',
                    'step_tokens', 'final_tokens', 'max_tokens'):
            if getattr(self, key) < 1:
                raise ValueError(key + ' must be positive')
        if self.retries < 0 or not 0 <= self.w1 <= 1:
            raise ValueError('invalid retries or w1')




In [ ]:
def numeric(value):
    return isinstance(value, (int, float)) and not isinstance(value, bool) and math.isfinite(value)


def safe_arithmetic(expr):
    """Small arithmetic language; reject prose and bound expensive operations."""
    if not isinstance(expr, str) or len(expr) > 256:
        return None
    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
           ast.Mod: operator.mod, ast.Pow: operator.pow}

    def evaluate(node):
        if isinstance(node, ast.Constant) and numeric(node.value):
            value = node.value
        elif isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
            value = evaluate(node.operand) * (-1 if isinstance(node.op, ast.USub) else 1)
        elif isinstance(node, ast.BinOp) and type(node.op) in ops:
            left, right = evaluate(node.left), evaluate(node.right)
            if isinstance(node.op, ast.Pow) and abs(right) > 12:
                raise ValueError('exponent limit')
            value = ops[type(node.op)](left, right)
        else:
            raise ValueError('unsupported arithmetic')
        if not numeric(value) or abs(value) > 1e15:
            raise ValueError('arithmetic limit')
        return value

    try:
        tree = ast.parse(expr.strip().replace('$', '').replace(',', ''), mode='eval')
        if sum(1 for _ in ast.walk(tree)) > 80:
            return None
        return float(evaluate(tree.body))
    except (ValueError, SyntaxError, ArithmeticError, TypeError, RecursionError):
        return None




In [ ]:
def equation_status(equation):
    if not equation or equation.lower() == 'none':
        return 'missing', None
    if equation.count('=') != 1:
        return 'unsupported', None
    lhs, rhs = map(safe_arithmetic, equation.split('='))
    if lhs is None or rhs is None:
        return 'unsupported', None
    return ('valid' if math.isclose(lhs, rhs, rel_tol=1e-9, abs_tol=1e-6) else 'invalid'), rhs


def equation_key(equation):
    try:
        return '|'.join(ast.dump(ast.parse(s.strip().replace('$', '').replace(',', ''), mode='eval'))
                        for s in equation.split('='))
    except SyntaxError:
        return equation.lower().replace(' ', '')




In [ ]:
def parse_object(raw):
    text = raw.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*|\s*```$', '', text)
    obj = json.loads(text)
    if not isinstance(obj, dict):
        raise ValueError('expected one JSON object')
    return obj


def validate_step(obj):
    if not set(STEP_SCHEMA['required']).issubset(obj) or not set(obj).issubset(STEP_SCHEMA['properties']):
        raise ValueError('missing or unexpected fields')
    if any(not isinstance(obj[k], str) for k in ('step', 'equation', 'target')):
        raise ValueError('text fields must be strings')
    if not isinstance(obj['done'], bool):
        raise ValueError('invalid done flag')
    expression = obj.get('expression', '').strip()
    result = obj.get('result')
    if expression or result is not None:
        computed = safe_arithmetic(expression)
        if computed is None or not numeric(result) or not math.isclose(
                computed, result, rel_tol=1e-9, abs_tol=1e-6):
            raise ValueError('expression_result_mismatch')
        obj = dict(obj, equation=f'{expression} = {result}')
    if not obj['step'].strip():
        if equation_status(obj['equation'])[0] != 'valid' or not obj['target'].strip():
            raise ValueError('empty step without verified calculation')
        obj = dict(obj, step='Computed ' + obj['target'].strip())
    if obj['done']:
        if not numeric(obj['answer']) or not obj['target'].strip():
            raise ValueError('completed answer needs a number and target')
    elif obj['answer'] is not None:
        raise ValueError('unfinished step must have null answer')
    # A chain is acceptable only when EVERY expression has the same numeric value.
    # Keep the original response in the request trace; normalize the validated object.
    if obj['equation'].count('=') > 1:
        parts = obj['equation'].split('=')
        if len(parts) > 8 or len(obj['equation']) > 512:
            raise ValueError('equation_chain_limit')
        values = [safe_arithmetic(part) for part in parts]
        if any(value is None for value in values):
            raise ValueError('equation_unsupported')
        if not all(math.isclose(values[0], value, rel_tol=1e-9, abs_tol=1e-6) for value in values[1:]):
            raise ValueError('equation_invalid')
        obj = dict(obj, equation=parts[0].strip() + ' = ' + parts[-1].strip())
    status, _ = equation_status(obj['equation'])
    if status in ('unsupported', 'invalid'):
        raise ValueError('equation_' + status)
    if not obj['done'] and status != 'valid':
        raise ValueError('arithmetic step needs a valid equation')
    return obj




In [ ]:
def extract_explicit_answer(text):
    """Only a complete final marker; never guess from intermediate numbers."""
    matches = re.findall(r'^\s*(?:\*\*)?FINAL ANSWER:\s*(?:\*\*)?\s*\$?\s*'
                         r'([+-]?(?:\d[\d,]*(?:\.\d+)?|\.\d+))\s*(?:\*\*)?\s*$',
                         text, re.MULTILINE | re.IGNORECASE)
    if len(matches) != 1:
        return None
    value = float(matches[0].replace(',', ''))
    return value if numeric(value) else None


class Solver:
    """backend(prompt, schema, temperature, seed, max_tokens) -> response dict.

    Response keys: text, prompt_tokens, completion_tokens, done_reason.
    All real requests including retries and terminal review count toward cost.
    """
    def __init__(self, backend, config=None):
        self.backend = backend
        self.cfg = config or Config()
        self.events = []
        self.calls = self.tokens = 0
        self.failure = None
        self.rejections = {}
        self.review_cache = {}

    def request(self, prompt, schema, temp, purpose, limit):
        feedback = ''
        for attempt in range(self.cfg.retries + 1):
            if self.calls >= self.cfg.max_calls:
                self.failure = 'call_limit'
                return None
            seed = (self.cfg.seed + self.calls) % (2**31)
            self.calls += 1
            event = {'kind': 'request', 'purpose': purpose, 'attempt': attempt,
                     'seed': seed, 'prompt': prompt + feedback, 'token_limit': limit,
                     'schema': schema, 'temperature': temp}
            self.events.append(event)
            try:
                response = self.backend(prompt + feedback, schema, temp, seed, limit)
                event.update(response)
                self.tokens += response.get('prompt_tokens', 0) + response.get('completion_tokens', 0)
                if response.get('done_reason') == 'length':
                    raise ValueError('truncated')
                raw = response['text']
                obj = parse_object(raw) if schema else extract_explicit_answer(raw)
                if schema == STEP_SCHEMA:
                    obj = validate_step(obj)
                elif schema == REVIEW_SCHEMA:
                    if (not isinstance(obj.get('valid'), bool) or not numeric(obj.get('answer'))
                            or not isinstance(obj.get('reason'), str)):
                        raise ValueError('invalid review')
                elif schema == JUDGE_SCHEMA:
                    if not numeric(obj.get('score')):
                        raise ValueError('invalid judge score')
                    score = obj['score']
                    if -1e-9 <= score <= 1 + 1e-9 and not 0 <= score <= 1:
                        obj['score'] = min(1.0, max(0.0, score))
                        event['score_clamped_from'] = score
                    elif not 0 <= score <= 1:
                        raise ValueError('invalid judge score')
                if obj is None:
                    raise ValueError('missing_final_answer')
                event['parsed'] = obj
                self.failure = None
                return obj
            except Exception as exc:
                event['error'] = str(exc)
                self.failure = str(exc)
                instruction = (
                    'Return the requested JSON search model with valid function definitions.'
                    if purpose == 'tos_model_synthesis' else
                    'Return calculations, result and target in the executable-equations JSON schema.'
                    if purpose == 'executable_equations' else
                    'Return valid (boolean), answer (number), and reason (string).'
                    if schema == REVIEW_SCHEMA else
                    'Return a score from zero to one in the JSON schema.'
                    if schema == JUDGE_SCHEMA else
                    'Return step, equation, done, answer and target. Include expression and numeric result '
                    'as separate fields; their arithmetic must agree. Use a numeric arithmetic equality; '
                    'done is a boolean and answer is null until the final quantity is solved.'
                    if schema == STEP_SCHEMA else
                    'Finish with FINAL ANSWER: <decimal number> on its own line.')
                feedback = ('\nPrevious response: ' + str(event.get('text', ''))[:2000]
                            + '\nYour previous response was rejected: ' + str(exc) + '. ' + instruction)
                if str(exc) == 'truncated':
                    limit = min(limit * 2, self.cfg.max_tokens)
        return None

    def terminal_review(self, question, steps, candidate):
        # No expected answer enters this prompt. This common check costs real tokens.
        key = (tuple(equation_key(step['equation']) for step in steps + [candidate]),
               candidate['answer'], candidate['target'].strip().lower())
        if key in self.review_cache:
            self.events.append({'kind': 'terminal_cache_hit', 'candidate': candidate})
            return self.review_cache[key]
        prompt = (f'Independently solve this problem: {question}\n'
                  f'Proposed reasoning: {json.dumps(steps + [candidate])}\n'
                  'Check the requested quantity, units, use of givens, and complete reasoning. '
                  'An intermediate subtotal, speed instead of distance, or number copied from '
                  'the question is insufficient. Return valid=true only if the proposed numeric '
                  'answer and target solve the whole problem. Supply your independently computed '
                  'answer and explain your decision. Do not merely check the last equation.')
        review = self.request(prompt, REVIEW_SCHEMA, 0.1, 'terminal_review', self.cfg.final_tokens)
        accepted = (review is not None and review['valid'] and
                    math.isclose(review['answer'], candidate['answer'], rel_tol=0, abs_tol=1e-4))
        if self.cfg.review_appeal and not accepted and self.calls < self.cfg.max_calls:
            # One independent appeal can recover a false negative. Both votes are logged.
            appeal = self.request(prompt + '\nRe-evaluate independently. Check every quantity '
                                  'and the final arithmetic before deciding.',
                                  REVIEW_SCHEMA, 0.1, 'terminal_review_appeal', self.cfg.final_tokens)
            accepted = (appeal is not None and appeal['valid'] and
                        math.isclose(appeal['answer'], candidate['answer'], rel_tol=0, abs_tol=1e-4))
        else:
            appeal = None
        self.events.append({'kind': 'terminal_decision', 'candidate': candidate,
                            'review': review, 'appeal': appeal, 'accepted': accepted})
        self.review_cache[key] = accepted
        return accepted

    def solve(self, question, method):
        if method == METHODS[5]:
            return solve_thought_of_search(self, question)
        started = time.time()
        answer, status, best = None, 'exhausted', []
        if method == METHODS[0]:
            # Exactly one baseline request: truncation is diagnosed, never silently retried.
            from dataclasses import replace
            previous = self.cfg
            self.cfg = replace(previous, retries=0)
            answer = self.request(
                f'Problem: {question}\nSolve fully with clear reasoning. Finish on a new line '
                'with exactly FINAL ANSWER: <number>. Use a decimal number without units '
                'on that line.', None, 0.1, 'standard_cot', self.cfg.max_tokens)
            self.cfg = previous
            status = 'done' if answer is not None else (self.failure or 'missing_final_answer')
        else:
            greedy = method == METHODS[1]
            judge = method == METHODS[2]
            width = 1 if greedy else self.cfg.beam
            n = 1 if greedy else self.cfg.candidates
            w3 = 0.1 if method == METHODS[3] else 0.0
            counter = itertools.count()
            heap = [(-1., next(counter), [])]
            seen_paths = set()
            best_quality = -1.
            expansions = children = 0
            stalled = {}
            while heap and expansions < self.cfg.max_expansions and self.calls < self.cfg.max_calls:
                neg, _, steps = heapq.heappop(heap)
                if len(steps) >= self.cfg.max_depth:
                    continue
                expansions += 1
                self.events.append({'kind': 'expand', 'priority': -neg, 'steps': steps,
                                    'depth': len(steps), 'expansion': expansions})
                history = '\n'.join(s['target'] + ': ' + s['equation'] for s in steps)
                prompt = (f'Problem: {question}\nSteps completed: {history}\n'
                          'Give ONE new arithmetic step. Use the step schema. equation must '
                          'contain only numbers, parentheses, arithmetic operators and one =, '
                          'e.g. 3 * 12 = 36. No units, variables or prose in equation. '
                          'Use // and arithmetic to express rounding when needed. '
                          'Do not repeat a previous calculation. done is a boolean. '
                          'Until the entire requested quantity is solved, answer=null and done=false. '
                          'When solved, done=true, answer is the numeric final answer, target '
                          'describes the requested quantity and units; equation may be empty '
                          'if an earlier step already calculated that answer. Set done=true in the '
                          'same response that computes the requested final quantity. Explain the result in step. '
                          'Include expression as a numeric arithmetic expression and result as its numeric value '
                          'for every computed step.')
                branch_key = json.dumps(steps, sort_keys=True)
                if branch_key in stalled:
                    prompt += '\nThis branch stalled: ' + stalled[branch_key]['reason']
                    prompt += '. Produce a valid new calculation, or mark the computed final answer done=true.'
                if branch_key in self.rejections:
                    prompt += '\nEarlier terminal answers were rejected: ' + json.dumps(self.rejections[branch_key])
                    prompt += '. Recheck the requested quantity and continue with a new calculation.'
                candidates = []
                for _ in range(n):
                    c = self.request(prompt, STEP_SCHEMA, 0.1 if greedy else 0.7,
                                     'generate', self.cfg.step_tokens)
                    if c is None:
                        continue
                    eq_state, rhs = equation_status(c['equation'])
                    repeated = not c['done'] and any(
                        c['step'].strip().lower() == s['step'].strip().lower() or
                        equation_key(c['equation']) == equation_key(s['equation'])
                        for s in steps)
                    if repeated or c in candidates:
                        self.events.append({'kind': 'rejected', 'reason': 'repeated_step', 'candidate': c})
                        # One bounded replacement request with explicit feedback.
                        c = self.request(prompt + '\nThe previous candidate repeated a calculation. '
                                         'Advance to a new calculation or declare the final answer.',
                                         STEP_SCHEMA, 0.7, 'repeat_recovery', self.cfg.step_tokens)
                        if c is None or c in candidates or (not c['done'] and any(
                            c['step'].strip().lower() == s['step'].strip().lower() or
                            equation_key(c['equation']) == equation_key(s['equation'])
                            for s in steps)):
                            continue
                    candidates.append(c)
                if not candidates:
                    attempt = stalled.get(branch_key, {}).get('attempts', 0)
                    reason = self.failure or 'all candidates repeated earlier calculations'
                    self.events.append({'kind': 'branch_stalled', 'reason': reason, 'steps': steps})
                    if attempt < self.cfg.retries:
                        stalled[branch_key] = {'attempts': attempt + 1, 'reason': reason}
                        heapq.heappush(heap, (neg, next(counter), steps))
                children += len(candidates)
                texts = [c['step'] + ' ' + c['target'] for c in candidates]
                relevance = lexical_relevance(texts, question)
                completed = []
                for c, t in zip(candidates, relevance):
                    eq_state, rhs = equation_status(c['equation'])
                    r = 1.0 if eq_state == 'valid' else 0.25
                    observed_output = [e['completion_tokens'] for e in self.events
                                       if e.get('purpose') == 'generate'
                                       and isinstance(e.get('completion_tokens'), (int, float))]
                    expected_output = (sum(observed_output) / len(observed_output)
                                       if observed_output else 192)
                    next_history = prompt + '\n' + c['target'] + ': ' + c['equation']
                    next_prompt = len(next_history.encode('utf-8')) / 3
                    cost = next_prompt + expected_output
                    score = r * (self.cfg.w1 + (1-self.cfg.w1)*t) / (1+w3*cost/500)
                    if judge:
                        rating = self.request(
                            f'Problem: {question}\nPrior steps: {json.dumps(steps)}\n'
                            f'Candidate: {json.dumps(c)}\nRate progress and correctness from 0 to 1.',
                            JUDGE_SCHEMA, 0.1, 'judge', self.cfg.step_tokens)
                        if rating is None:
                            continue
                        score = rating['score']
                    self.events.append({'kind': 'candidate_score', 'candidate': c, 'depth': len(steps),
                                        'equation_status': eq_state, 'r': r, 't': t,
                                        'cost': cost, 'score': score})
                    if c['done']:
                        # Final numeric answer must be grounded in a verified calculation.
                        values = [equation_status(s['equation'])[1] for s in steps + [c]]
                        grounded = any(v is not None and math.isclose(v, c['answer'], abs_tol=1e-4,
                                                                    rel_tol=0) for v in values)
                        if grounded and self.terminal_review(question, steps, c):
                            completed.append((score, c, steps + [c]))
                        else:
                            self.events.append({'kind': 'rejected', 'reason': 'terminal_unverified',
                                                'candidate': c, 'grounded': grounded})
                            reasons = self.rejections.setdefault(branch_key, [])
                            reasons.append(c)
                            # Revisit this branch at most twice; global budgets still apply.
                            if len(reasons) <= self.cfg.retries:
                                heapq.heappush(heap, (-score, next(counter), steps))
                        continue
                    path = steps + [c]
                    if score > best_quality:
                        best_quality, best = score, path
                    signature = tuple(equation_key(s['equation']) for s in path)
                    if signature not in seen_paths:
                        seen_paths.add(signature)
                        heapq.heappush(heap, (-score, next(counter), path))
                if completed:
                    _, chosen, best = max(completed, key=lambda item: item[0])
                    answer, status = chosen['answer'], 'done'
                    break
                heap = heapq.nsmallest(width, heap)
                heapq.heapify(heap)
            if answer is None:
                status = ('call_limit' if self.calls >= self.cfg.max_calls else
                          'expansion_limit' if expansions >= self.cfg.max_expansions else
                          'depth_limit' if best and len(best) >= self.cfg.max_depth else 'exhausted')
        return {'method': method, 'ans': answer, 'status': status, 'steps': best,
                'calls': self.calls, 'tokens': self.tokens, 'sec': time.time()-started,
                'trace': self.events, 'last_error': self.failure, 'seed': self.cfg.seed,
                'config': vars(self.cfg), 'engine_version': VERSION}




In [ ]:
def lexical_relevance(texts, question):
    if not texts:
        return []
    # Smooth IDF and L2 normalization, equivalent to the original vectorizer's
    # default weighting, using only stdlib so offline regressions need no model.
    from collections import Counter
    docs = [Counter(re.findall(r'\b\w\w+\b', text.lower())) for text in texts + [question]]
    frequency = Counter(word for doc in docs for word in doc)
    vectors = []
    for doc in docs:
        weighted = {word: count * (1 + math.log((1+len(docs))/(1+frequency[word])))
                    for word, count in doc.items()}
        norm = math.sqrt(sum(weight**2 for weight in weighted.values())) or 1
        vectors.append({word: weight/norm for word, weight in weighted.items()})
    goal = vectors[-1]
    scores = [sum(weight * goal.get(word, 0) for word, weight in vec.items()) for vec in vectors[:-1]]
    lo, hi = min(scores), max(scores)
    return [(s-lo)/(hi-lo) for s in scores] if hi-lo > 1e-9 else [0.5]*len(texts)




In [ ]:
def evaluate(backend, question, expected, method, config=None, problem_id=None):
    if method not in METHODS:
        raise ValueError('unknown method')
    result = Solver(backend, config).solve(question, method)
    ans = result['ans']
    correct = ans is not None and math.isclose(ans, expected, rel_tol=0, abs_tol=1e-4)
    result.update(question=question, expected=expected, correct=correct, problem_id=problem_id,
                  outcome='missing' if ans is None else 'correct' if correct else 'wrong',
                  final_answer=ans, expected_answer=expected)
    return result


## Embedded source: tos_search


In [ ]:
"""Thought of Search arithmetic adaptation: synthesize components, then local BFS.

Reference: Katz et al., https://arxiv.org/abs/2404.11833.
The original human-reviewed, reusable domain code is replaced here by bounded
per-problem synthesis of restricted Python functions. No paper-level soundness
or completeness guarantee is claimed for the inferred word-problem model.
"""
import ast
import copy
from collections import deque
import json
import math
import operator
import time


TOS_METHOD = 'Thought of Search (Arithmetic Adaptation)'
TOS_SCHEMA = {
    'type': 'object', 'additionalProperties': False,
    'properties': {
        'initial_state': {'type': 'object', 'additionalProperties': {'type': 'number'}},
        'successor_code': {'type': 'string'}, 'goal_code': {'type': 'string'},
        'answer_code': {'type': 'string'}, 'target': {'type': 'string'},
        'objective': {'type': 'string', 'enum': ['first', 'min', 'max']},
    },
    'required': ['initial_state', 'successor_code', 'goal_code', 'answer_code', 'target', 'objective'],
}




In [ ]:
class RestrictedFunction:
    """Interpret bounded arithmetic Python; never exec/eval generated code."""
    allowed = (ast.Expression, ast.Constant, ast.Name, ast.Load, ast.Store,
               ast.Subscript, ast.Dict, ast.List, ast.Tuple, ast.BinOp, ast.UnaryOp,
               ast.BoolOp, ast.Compare, ast.IfExp, ast.ListComp, ast.comprehension,
               ast.Call, ast.Add, ast.Sub, ast.Mult, ast.Div, ast.FloorDiv, ast.Mod,
               ast.Pow, ast.USub, ast.UAdd, ast.Not, ast.And, ast.Or, ast.Eq,
               ast.NotEq, ast.Lt, ast.LtE, ast.Gt, ast.GtE, ast.BitOr,
               ast.Assign, ast.AugAssign, ast.If, ast.For, ast.Return, ast.Expr,
               ast.Break, ast.Continue, ast.Pass, ast.Attribute, ast.keyword)
    functions = {'abs', 'min', 'max', 'sum', 'len', 'round', 'ceil', 'floor', 'range',
                 'dict', 'int', 'float'}
    methods = {'copy', 'get', 'update', 'append'}

    def __init__(self, code, name):
        if not isinstance(code, str) or len(code) > 12000:
            raise ValueError('component code must be a string of at most 12000 characters')
        tree = ast.parse(code)
        if len(tree.body) != 1 or not isinstance(tree.body[0], ast.FunctionDef):
            raise ValueError('component must define exactly one function')
        function = tree.body[0]
        args = function.args
        if (function.name != name or function.decorator_list or function.returns
                or len(args.args) != 1 or args.args[0].arg != 'state' or args.args[0].annotation
                or args.posonlyargs or args.kwonlyargs or args.vararg or args.kwarg or args.defaults
                or args.kw_defaults):
            raise ValueError('use def ' + name + '(state): without decorators, defaults, or annotations')
        self.body = function.body
        nodes = [node for statement in self.body for node in ast.walk(statement)]
        if not any(isinstance(node, ast.Return) for node in nodes):
            raise ValueError('function requires a return statement')
        if len(nodes) > 1000:
            raise ValueError('component AST is too large')
        for node in nodes:
            if not isinstance(node, self.allowed):
                raise ValueError('unsupported syntax: ' + type(node).__name__)
            if isinstance(node, ast.Name) and node.id.startswith('_'):
                raise ValueError('private names are unsupported')
            if isinstance(node, ast.Attribute) and node.attr not in self.methods:
                raise ValueError('unsupported method or attribute: ' + node.attr)
            if isinstance(node, ast.Call):
                helper = isinstance(node.func, ast.Name) and node.func.id in self.functions
                method = isinstance(node.func, ast.Attribute) and node.func.attr in self.methods
                if not (helper or method) or any(isinstance(arg, ast.Starred) for arg in node.args):
                    raise ValueError('only arithmetic helpers and bounded collection methods are supported')
                if any(keyword.arg is None for keyword in node.keywords):
                    raise ValueError('keyword unpacking is unsupported')
                if node.keywords and not (helper and node.func.id == 'dict'):
                    raise ValueError('only dict accepts keyword arguments')
            if isinstance(node, ast.comprehension) and (
                    node.is_async or not isinstance(node.target, ast.Name)
                    or node.target.id == 'state' or node.target.id in self.functions):
                raise ValueError('comprehensions need a simple local variable')
        if not any(isinstance(node, ast.Name) and node.id == 'state' for node in nodes):
            raise ValueError('component must depend on state, not a constant answer')

    def __call__(self, state):
        self.remaining = 20000
        # Generated assignments cannot mutate states retained in the BFS queue or trace.
        outcome = self.execute(self.body, {'state': copy.deepcopy(state)})
        if outcome is None or outcome[0] != 'return':
            raise ValueError('function reached its end without returning')
        return outcome[1]

    def tick(self):
        self.remaining -= 1
        if self.remaining < 0:
            raise ValueError('expression operation limit')

    def assign(self, target, value, env):
        if isinstance(target, ast.Name) and target.id not in self.functions and not target.id.startswith('_'):
            env[target.id] = value
        elif isinstance(target, ast.Subscript):
            collection = self.visit(target.value, env)
            key = self.visit(target.slice, env)
            if not isinstance(collection, (dict, list)):
                raise ValueError('assignment needs a local collection')
            collection[key] = value
            if len(collection) > 256:
                raise ValueError('collection limit')
        else:
            raise ValueError('unsupported assignment target')

    def execute(self, statements, env):
        for statement in statements:
            self.tick()
            if isinstance(statement, ast.Return):
                return 'return', self.visit(statement.value, env) if statement.value else None
            if isinstance(statement, ast.Assign):
                value = self.visit(statement.value, env)
                for target in statement.targets:
                    self.assign(target, value, env)
            elif isinstance(statement, ast.AugAssign):
                expression = ast.BinOp(left=statement.target, op=statement.op, right=statement.value)
                self.assign(statement.target, self.visit(expression, env), env)
            elif isinstance(statement, ast.If):
                branch = statement.body if self.visit(statement.test, env) else statement.orelse
                outcome = self.execute(branch, env)
                if outcome:
                    return outcome
            elif isinstance(statement, ast.For):
                values = self.visit(statement.iter, env)
                if not isinstance(values, list) or len(values) > 256:
                    raise ValueError('for requires a bounded list or range')
                broke = False
                for value in list(values):
                    self.tick()
                    self.assign(statement.target, value, env)
                    outcome = self.execute(statement.body, env)
                    if outcome and outcome[0] == 'return':
                        return outcome
                    if outcome and outcome[0] == 'break':
                        broke = True
                        break
                if not broke:
                    outcome = self.execute(statement.orelse, env)
                    if outcome:
                        return outcome
            elif isinstance(statement, ast.Break):
                return 'break', None
            elif isinstance(statement, ast.Continue):
                return 'continue', None
            elif isinstance(statement, ast.Expr):
                self.visit(statement.value, env)
            elif not isinstance(statement, ast.Pass):
                raise ValueError('unsupported statement')

    def visit(self, node, env):
        self.tick()
        if isinstance(node, ast.Constant):
            value = node.value
            if not isinstance(value, (str, bool)) and not numeric(value):
                raise ValueError('unsupported constant')
            if isinstance(value, str) and len(value) > 64:
                raise ValueError('string limit')
        elif isinstance(node, ast.Name):
            if node.id not in env:
                raise ValueError('undefined local variable: ' + node.id)
            value = env[node.id]
        elif isinstance(node, ast.Subscript):
            collection, key = self.visit(node.value, env), self.visit(node.slice, env)
            try:
                value = collection[key]
            except KeyError as exc:
                raise ValueError(f'Undefined state key {key!r}; dictionary literals read the OLD state. '
                                 'Compute dependent fields with sequential assignments or separate stages.') from exc
        elif isinstance(node, ast.Dict):
            if len(node.keys) > 32:
                raise ValueError('state size limit')
            value = {}
            for key, child in zip(node.keys, node.values):
                if key is None:
                    mapping = self.visit(child, env)
                    if not isinstance(mapping, dict):
                        raise ValueError('dictionary unpacking needs a dictionary')
                    value.update(mapping)
                else:
                    value[self.visit(key, env)] = self.visit(child, env)
        elif isinstance(node, (ast.List, ast.Tuple)):
            if len(node.elts) > 256:
                raise ValueError('collection limit')
            value = [self.visit(item, env) for item in node.elts]
        elif isinstance(node, ast.BinOp):
            a, b = self.visit(node.left, env), self.visit(node.right, env)
            if isinstance(node.op, ast.BitOr):
                if not isinstance(a, dict) or not isinstance(b, dict):
                    raise ValueError('union is supported only for dictionaries')
                value = dict(a, **b)
                if len(value) > 32:
                    raise ValueError('state size limit')
                return value
            if not numeric(a) or not numeric(b):
                raise ValueError('arithmetic requires numbers')
            if isinstance(node.op, ast.Pow) and abs(b) > 12:
                raise ValueError('exponent limit')
            operations = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
                          ast.Div: operator.truediv, ast.FloorDiv: operator.floordiv,
                          ast.Mod: operator.mod, ast.Pow: operator.pow}
            value = operations[type(node.op)](a, b)
        elif isinstance(node, ast.UnaryOp):
            a = self.visit(node.operand, env)
            if isinstance(node.op, ast.Not):
                value = not a
            elif numeric(a):
                value = -a if isinstance(node.op, ast.USub) else a
            else:
                raise ValueError('unary arithmetic requires a number')
        elif isinstance(node, ast.BoolOp):
            value = self.visit(node.values[0], env)
            for child in node.values[1:]:
                if (isinstance(node.op, ast.And) and not value) or (isinstance(node.op, ast.Or) and value):
                    break
                value = self.visit(child, env)
        elif isinstance(node, ast.Compare):
            a = self.visit(node.left, env)
            operations = {ast.Eq: operator.eq, ast.NotEq: operator.ne, ast.Lt: operator.lt,
                          ast.LtE: operator.le, ast.Gt: operator.gt, ast.GtE: operator.ge}
            value = True
            for op, child in zip(node.ops, node.comparators):
                b = self.visit(child, env)
                if not operations[type(op)](a, b):
                    value = False
                    break
                a = b
        elif isinstance(node, ast.IfExp):
            value = self.visit(node.body if self.visit(node.test, env) else node.orelse, env)
        elif isinstance(node, ast.ListComp):
            value = []
            def collect(index, scope):
                if index == len(node.generators):
                    value.append(self.visit(node.elt, scope))
                    if len(value) > 256:
                        raise ValueError('collection limit')
                    return
                gen = node.generators[index]
                iterable = self.visit(gen.iter, scope)
                if not isinstance(iterable, list) or len(iterable) > 256:
                    raise ValueError('bounded list required')
                for item in iterable:
                    self.remaining -= 1
                    if self.remaining < 0:
                        raise ValueError('expression operation limit')
                    local = dict(scope, **{gen.target.id: item})
                    if all(self.visit(condition, local) for condition in gen.ifs):
                        collect(index + 1, local)
            collect(0, env)
        elif isinstance(node, ast.Call):
            args = [self.visit(arg, env) for arg in node.args]
            if isinstance(node.func, ast.Attribute):
                receiver = self.visit(node.func.value, env)
                method = node.func.attr
                if isinstance(receiver, dict) and method == 'copy' and not args:
                    return dict(receiver)
                if isinstance(receiver, dict) and method == 'get' and 1 <= len(args) <= 2:
                    return receiver.get(*args)
                if isinstance(receiver, dict) and method == 'update' and len(args) == 1 and isinstance(args[0], dict):
                    receiver.update(args[0])
                elif isinstance(receiver, list) and method == 'append' and len(args) == 1:
                    receiver.append(args[0])
                else:
                    raise ValueError('unsupported collection method arguments')
                if len(receiver) > 256:
                    raise ValueError('collection limit')
                return None
            name = node.func.id
            if name == 'range':
                if not 1 <= len(args) <= 3 or any(not numeric(arg) or int(arg) != arg for arg in args):
                    raise ValueError('range needs integer bounds')
                sequence = range(*(int(arg) for arg in args))
                if len(sequence) > 256:
                    raise ValueError('range limit')
                value = list(sequence)
            elif name == 'dict':
                if len(args) > 1 or (args and not isinstance(args[0], dict)):
                    raise ValueError('dict needs at most one dictionary')
                value = dict(args[0]) if args else {}
                value.update({keyword.arg: self.visit(keyword.value, env) for keyword in node.keywords})
            else:
                funcs = {'abs': abs, 'min': min, 'max': max, 'sum': sum, 'len': len,
                         'round': round, 'ceil': math.ceil, 'floor': math.floor, 'int': int, 'float': float}
                value = funcs[name](*args)
        else:
            raise ValueError('unsupported expression')
        if isinstance(value, (dict, list, tuple)) and len(value) > 256:
            raise ValueError('collection limit')
        if isinstance(value, (int, float)) and not isinstance(value, bool):
            if not numeric(value) or abs(value) > 1e15:
                raise ValueError('numeric limit')
        return value




In [ ]:
def validate_state(state, keys=None):
    if (not isinstance(state, dict) or not 1 <= len(state) <= 32
            or any(not isinstance(key, str) or not key or len(key) > 64 for key in state)
            or any(not numeric(value) or abs(value) > 1e15 for value in state.values())):
        raise ValueError('state must have string keys and finite numeric values')
    return state


def validate_model(model):
    if set(model) != set(TOS_SCHEMA['required']):
        raise ValueError('missing or unexpected search-model fields')
    initial = validate_state(model['initial_state'])
    if model['objective'] not in ('first', 'min', 'max') or not isinstance(model['target'], str):
        raise ValueError('invalid objective or target')
    if not model['target'].strip():
        raise ValueError('target is required')
    successor = RestrictedFunction(model['successor_code'], 'successors')
    goal = RestrictedFunction(model['goal_code'], 'is_goal')
    answer = RestrictedFunction(model['answer_code'], 'answer')
    initial_goal = goal(initial)
    if not isinstance(initial_goal, bool):
        raise ValueError('goal test must return a boolean')
    children = [] if initial_goal else successor(initial)
    if not isinstance(children, list) or len(children) > 256:
        raise ValueError('initial successors must be a bounded list')
    for child in children:
        validate_state(child, set(initial))
    return initial, successor, goal, answer




In [ ]:
def local_bfs(model, config, events):
    initial, successors, is_goal, answer = validate_model(model)
    queue = deque([(initial, [])])
    visited = {json.dumps(initial, sort_keys=True)}
    expansions = 0
    best = None
    depth_cutoff = False
    while queue:
        state, path = queue.popleft()
        goal = is_goal(state)
        if not isinstance(goal, bool):
            raise ValueError('goal test must return a boolean')
        events.append({'kind': 'tos_visit', 'state': state, 'depth': len(path), 'goal': goal})
        if goal:
            value = answer(state)
            if not numeric(value):
                raise ValueError('answer function must return a finite number')
            candidate = (value, path)
            if model['objective'] == 'first':
                return value, path, 'done', expansions
            if best is None or (value < best[0] if model['objective'] == 'min' else value > best[0]):
                best = candidate
            continue
        if len(path) >= config.max_depth:
            depth_cutoff = True
            continue
        if expansions >= config.max_expansions:
            return None, [], 'expansion_limit', expansions
        expansions += 1
        events.append({'kind': 'tos_expand', 'state': state, 'depth': len(path)})
        children = successors(state)
        if not isinstance(children, list) or len(children) > 256:
            raise ValueError('successor function must return a bounded list')
        for child in children:
            validate_state(child, set(initial))
            key = json.dumps(child, sort_keys=True)
            if key not in visited:
                visited.add(key)
                queue.append((child, path + [{'state': child}]))
        if len(visited) > config.max_expansions * 256 + 1:
            raise ValueError('state budget exceeded')
    if depth_cutoff:
        return None, [], 'depth_limit', expansions
    if best is not None:
        return best[0], best[1], 'done', expansions
    return None, [], 'exhausted', expansions




In [ ]:
def solve_thought_of_search(solver, question):
    started = time.time()
    example = {
        'initial_state': {'bags': 2, 'each': 5, 'removed': 3, 'stage': 0, 'value': 0},
        'successor_code': (
            'def successors(state):\n'
            '    return [{"bags":state["bags"], "each":state["each"], '
            '"removed":state["removed"], "stage":state["stage"]+1, '
            '"value":state["bags"]*state["each"] if state["stage"]==0 '
            'else state["value"]-state["removed"]}] if state["stage"]<2 else []'),
        'goal_code': 'def is_goal(state):\n    return state["stage"] == 2',
        'answer_code': 'def answer(state):\n    return state["value"]',
        'target': 'beads remaining', 'objective': 'first',
    }
    prompt = (
        f'Problem: {question}\nImplement a symbolic arithmetic search model BEFORE searching. '
        'Return the JSON search-model schema. initial_state contains ONLY given numeric facts '
        'and numeric counters. States may add or omit keys, but preserve facts needed by later steps. '
        'target describes the requested quantity and units. successor_code defines def successors(state) '
        'returning a list of successor state dictionaries. goal_code defines def is_goal(state) '
        'returning a boolean. answer_code defines def answer(state) extracting the computed answer. '
        'Use ordinary local assignments, if/elif/else, bounded for loops, return, arithmetic, '
        'comparisons, conditional expressions and list comprehensions. Helpers: range, abs, min, '
        'max, sum, len, round, ceil, floor, int, float, dict. Methods: dict.copy/get/update and '
        'list.append. Dictionary union (state | updates) and {**state, ...} are supported. '
        'No imports, while loops, recursion, arbitrary attributes or external calls. '
        'range bounds must be integral, with at most 256 values. '
        'Do NOT put a guessed final answer in initial_state or hardcode a solution in the goal '
        'test. Compute through state transitions. A dictionary literal reads the OLD state: '
        'new fields in it cannot read one another. Use sequential local assignments or separate '
        'stages for dependent calculations. Advance stage only when its arithmetic is computed. '
        'Goal tests must not fire before all requested arithmetic is complete. '
        'Use explicit units and conversions from the question; document needed conversion factors '
        'as named numeric state fields, not unexplained result constants. '
        'For arithmetic chains, use an integer stage and compute the next result from state '
        'facts at each transition. For constraints, enumerate feasible choices and test them. '
        'For minimum/maximum questions use objective="min"/"max" and enumerate ALL feasible '
        'goal states, answer returning their objective value. Otherwise objective="first". '
        f'The local BFS has depth {solver.cfg.max_depth} and {solver.cfg.max_expansions} expansions. '
        'Example FORMAT for a different problem, 2 bags of 5 beads then remove 3: '
        + json.dumps(example) + '. '
        'Use the actual problem facts, not the example numbers.')
    result = None
    status = 'model_error'
    expansions = 0
    model = None
    # Repair structural/runtime failures and unsuccessful searches, never reference-answer feedback.
    for repair in range(solver.cfg.retries + 1):
        model = solver.request(prompt, TOS_SCHEMA, 0.1, 'tos_model_synthesis', solver.cfg.max_tokens)
        if model is None:
            break
        attempt_events = []
        try:
            result, path, status, expansions = local_bfs(model, solver.cfg, attempt_events)
            solver.events.extend(attempt_events)
            solver.events.append({'kind': 'tos_search_attempt', 'repair': repair,
                                  'status': status, 'expansions': expansions, 'answer': result})
            if result is not None:
                if solver.cfg.audit_tos:
                    audit_prompt = (
                        f'Problem: {question}\nGenerated search model: {json.dumps(model)}\n'
                        f'Visited solution path: {json.dumps(path)}\nComputed answer: {result}.\n'
                        'Independently verify that the initial facts, number of stages, state '
                        'transitions, requested target and units match the question. Compute '
                        'the answer independently. Return valid=false if the model skips or '
                        'duplicates a stage, uses an unsupported assumption, or returns the '
                        'wrong quantity. Give the independent numeric answer and reason.')
                    audit = solver.request(audit_prompt, REVIEW_SCHEMA, 0.1,
                                           'tos_semantic_audit', solver.cfg.final_tokens)
                    accepted = (audit is not None and audit['valid'] and
                                math.isclose(audit['answer'], result, rel_tol=0, abs_tol=1e-4))
                    solver.events.append({'kind': 'tos_audit', 'review': audit,
                                          'computed_answer': result, 'accepted': accepted})
                    if not accepted:
                        solver.failure = 'tos_semantic_audit_rejected'
                        result = None
                        status = 'semantic_unverified'
                        prompt += ('\nPrevious model: ' + json.dumps(model) +
                                   '\nIndependent audit rejected the model: ' +
                                   str(audit.get('reason') if audit else 'no valid audit') +
                                   '. Repair the transitions and target without using a '
                                   'guessed final answer.')
                        continue
                break
            solver.failure = 'search_' + status
            feedback = ('Search ended with ' + status + '. Check successor progress, goal reachability, '
                        'and whether unrelated counters consume the depth budget. Do not change the '
                        'problem to fit a guessed answer. Use aggregate arithmetic instead of '
                        'enumerating time steps if the stated depth budget is insufficient.')
            prompt += '\nPrevious model: ' + json.dumps(model) + '\nLocal failure: ' + feedback
            prompt += '\nLast visited states: ' + json.dumps(attempt_events[-3:])
        except (ValueError, TypeError, KeyError, SyntaxError, ArithmeticError, IndexError,
                RecursionError) as exc:
            solver.events.extend(attempt_events)
            solver.failure = str(exc)
            status = 'model_error'
            solver.events.append({'kind': 'tos_model_rejected', 'error': str(exc), 'repair': repair})
            prompt += '\nPrevious model: ' + json.dumps(model) + '\nLocal failure: ' + str(exc)
            prompt += '. Repair the model while preserving the problem semantics.'
    if solver.calls >= solver.cfg.max_calls and result is None:
        status = 'call_limit'
    return {'method': TOS_METHOD, 'ans': result, 'status': status,
            'steps': path if result is not None else [], 'calls': solver.calls,
            'tokens': solver.tokens, 'sec': time.time()-started, 'trace': solver.events,
            'last_error': solver.failure, 'seed': solver.cfg.seed, 'config': vars(solver.cfg),
            'engine_version': '4', 'baseline_version': 'tos-arithmetic-3' if solver.cfg.audit_tos
            else 'tos-arithmetic-2',
            'search_model': model,
            'local_expansions': sum(event['kind'] == 'tos_expand' for event in solver.events),
            'adaptation': 'Per-problem restricted Python components; local BFS; no human review '
                          'or LLM calls within BFS; bounded Python interpreter; structural/runtime '
                          'and unsuccessful-search repair calls are counted.'}


## Embedded source: benchmark_data


In [ ]:
"""Official GSM8K/SVAMP snapshots and deterministic, disjoint benchmark subsets.

Sampling depends only on IDs and a fixed selection seed, never model outcomes.
Cache hashes and selected records are exported so results can be reproduced.
"""
import argparse
import hashlib
import json
import math
from pathlib import Path
import re
import urllib.request

SOURCES = {
    'gsm8k_train': {
        'url': 'https://raw.githubusercontent.com/openai/grade-school-math/master/'
               'grade_school_math/data/train.jsonl',
        'dataset': 'gsm8k', 'source_split': 'train', 'format': 'jsonl', 'count': 7473,
        'reference': 'https://arxiv.org/abs/2110.14168',
        'license': 'MIT (see upstream repository)',
    },
    'gsm8k_test': {
        'url': 'https://raw.githubusercontent.com/openai/grade-school-math/master/'
               'grade_school_math/data/test.jsonl',
        'dataset': 'gsm8k', 'source_split': 'test', 'format': 'jsonl', 'count': 1319,
        'reference': 'https://arxiv.org/abs/2110.14168',
        'license': 'MIT (see upstream repository)',
    },
    'svamp': {
        'url': 'https://raw.githubusercontent.com/arkilpatel/SVAMP/main/SVAMP.json',
        'dataset': 'svamp', 'source_split': 'challenge', 'format': 'json', 'count': 1000,
        'reference': 'https://arxiv.org/abs/2103.07191',
        'license': 'MIT (see upstream repository)',
    },
}
SELECTION_SEED = 203
# Freeze the verified upstream bytes, including when Kaggle downloads afresh.
SOURCE_SHA256 = {
    'gsm8k_train': '17f347dc51477c50d4efb83959dbb7c56297aba886e5544ee2aaed3024813465',
    'gsm8k_test': '3730d312f6e3440559ace48831e51066acaca737f6eabec99bccb9e4b3c39d14',
    'svamp': '5be77703a6d891ae476d7c082787ad361392aa02453b132516cdd5f4e7934e3e',
}




In [ ]:
def parse_reference(value):
    """Reference-only parser, never applied to model output or sent to solvers."""
    if isinstance(value, bool):
        raise ValueError('boolean reference answer')
    text = str(value).strip()
    if '####' in text:
        text = text.rsplit('####', 1)[1].strip()
    text = text.replace(',', '').replace('$', '').strip()
    if not re.fullmatch(r'[+-]?(?:\d+(?:\.\d+)?|\.\d+)', text):
        raise ValueError('unsupported reference answer: ' + text)
    answer = float(text)
    if not math.isfinite(answer):
        raise ValueError('nonfinite reference answer')
    return answer




In [ ]:
def fingerprint(question):
    normalized = ' '.join(question.lower().split())
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()


def normalize_records(source, records):
    spec = SOURCES[source]
    normalized = []
    identities = set()
    for index, row in enumerate(records):
        if spec['dataset'] == 'gsm8k':
            question = row['question'].strip()
            expected = parse_reference(row['answer'])
            original_id = str(index)
        else:
            question = (row['Body'].strip() + ' ' + row['Question'].strip()).strip()
            expected = parse_reference(row['Answer'])
            original_id = str(row['ID'])
        source_id = f'{spec["dataset"]}:{spec["source_split"]}:{original_id}'
        if not question or source_id in identities:
            raise ValueError('empty question or duplicate source ID')
        identities.add(source_id)
        normalized.append(dict(question=question, expected=expected,
                               dataset=spec['dataset'], source_split=spec['source_split'],
                               source_id=source_id, question_sha256=fingerprint(question)))
    return normalized




In [ ]:
def read_snapshot(source, cache_dir, download=True):
    spec = SOURCES[source]
    cache = Path(cache_dir)
    cache.mkdir(parents=True, exist_ok=True)
    path = cache / (source + '.' + spec['format'])
    metadata_path = cache / (source + '.metadata.json')
    if not path.exists():
        if not download:
            raise FileNotFoundError('Missing dataset snapshot: ' + str(path))
        if metadata_path.exists():
            raise ValueError('Snapshot missing but provenance exists; restore it or use a new cache directory')
        request = urllib.request.Request(spec['url'], headers={'User-Agent': 'CostAwareBenchmark/3.1'})
        with urllib.request.urlopen(request, timeout=60) as response:
            raw = response.read(20_000_001)
        if len(raw) > 20_000_000:
            raise ValueError('unexpectedly large dataset download')
        if hashlib.sha256(raw).hexdigest() != SOURCE_SHA256[source]:
            raise ValueError('Upstream dataset changed; review it before changing the frozen source hash')
        records = (json.loads(raw) if spec['format'] == 'json' else
                   [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()])
        # Validate before caching a potentially truncated or wrong download.
        normalize_records(source, records)
        if len(records) != spec['count']:
            raise ValueError('Unexpected upstream dataset size for ' + source)
        metadata = dict(spec, sha256=hashlib.sha256(raw).hexdigest(), rows=len(records))
        with path.open('xb') as handle:
            handle.write(raw)
        with metadata_path.open('x', encoding='utf-8') as handle:
            json.dump(metadata, handle, indent=2)
    raw = path.read_bytes()
    metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
    actual_hash = hashlib.sha256(raw).hexdigest()
    if (metadata['sha256'] != actual_hash or actual_hash != SOURCE_SHA256[source]
            or metadata['url'] != spec['url']):
        raise ValueError('Dataset snapshot provenance/hash mismatch: ' + str(path))
    records = (json.loads(raw) if spec['format'] == 'json' else
               [json.loads(line) for line in raw.decode('utf-8').splitlines() if line.strip()])
    if len(records) != spec['count']:
        raise ValueError('Unexpected cached dataset size')
    return normalize_records(source, records), metadata




In [ ]:
def fixed_sample(records, count, seed, excluded=()):
    if count < 0:
        raise ValueError('sample count must be nonnegative')
    used = set(excluded)
    ordered = sorted(records, key=lambda row: hashlib.sha256(
        f'{seed}:{row["source_id"]}'.encode('utf-8')).hexdigest())
    result = []
    for row in ordered:
        if row['question_sha256'] not in used and len(result) < count:
            used.add(row['question_sha256'])
            result.append(dict(row))
    if len(result) != count:
        raise ValueError(f'Requested {count} distinct problems, only {len(result)} available')
    return result




In [ ]:
def build_subsets(catalog, selection_seed=SELECTION_SEED, dev_count=30, gsm_count=100, svamp_count=50):
    # Exclude all GSM8K test questions from development, not just the sampled test subset.
    test_fingerprints = {row['question_sha256'] for row in catalog['gsm8k_test']}
    development = fixed_sample(catalog['gsm8k_train'], dev_count, selection_seed, test_fingerprints)
    used = {row['question_sha256'] for row in development}
    gsm = fixed_sample(catalog['gsm8k_test'], gsm_count, selection_seed, used)
    used.update(row['question_sha256'] for row in gsm)
    svamp = fixed_sample(catalog['svamp'], svamp_count, selection_seed, used)
    for row in development:
        row['evaluation_split'] = 'development'
    for row in gsm + svamp:
        row['evaluation_split'] = 'evaluation'
    return {'development': development, 'evaluation': gsm + svamp,
            'pilot': development[:min(20, len(development))]}




In [ ]:
def prepare_benchmarks(cache_dir='kaggle/data/official', selection_seed=SELECTION_SEED, download=True):
    catalog, provenance = {}, {}
    for source in SOURCES:
        catalog[source], provenance[source] = read_snapshot(source, cache_dir, download)
    subsets = build_subsets(catalog, selection_seed)
    return subsets, {'selection_seed': selection_seed, 'sources': provenance,
                     'sampling': 'SHA256 rank of selection seed and source ID; exact question deduplication',
                     'development': '30 GSM8K train; excludes all exact GSM8K test matches',
                     'evaluation': '100 GSM8K test + 50 SVAMP challenge; no development overlap',
                     'pilot': 'first 20 selected development questions; not final evaluation'}


def export_subsets(output, subsets, provenance):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    for name, records in subsets.items():
        with (output / (name + '.jsonl')).open('x', encoding='utf-8') as handle:
            for record in records:
                handle.write(json.dumps(record, ensure_ascii=False) + '\n')
    (output / 'dataset_manifest.json').write_text(json.dumps(provenance, indent=2), encoding='utf-8')




## Embedded source: word_bank_experiment


In [ ]:
"""Sources embedded as readable cells in Word_bank_approch.ipynb.

The notebook builder removes repository imports. No inference runs on import.
"""
import ast
import hashlib
import keyword
import json
import math
import re
import time
from collections import Counter
from dataclasses import replace
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer



NEW_METHODS = (
    'Local Search: No Lexical Features',
    'Local Search: Question TF-IDF',
    'Local Search: TF-IDF + Word Bank',
    'Local Search: Word Bank + Compact State',
)
TREE_METHODS = (METHODS[0], METHODS[2], METHODS[3], METHODS[4], METHODS[5])
ALL_METHODS = TREE_METHODS + NEW_METHODS
EXPERIMENT_VERSION = 'word-bank-search-3'




In [ ]:
def lexical_text(text):
    # Remove numeric answers before fitting/ranking. Keep words and phrases.
    text = re.sub(r'<<.*?>>', ' ', text)
    return re.sub(r'\d+(?:[.,]\d+)*', ' ', text.lower())


class ReferenceWordBank:
    """Question-conditioned phrase bank from gold TRAINING solutions only.

    This is positive-reference similarity, not a supervised correctness model.
    TF-IDF is fitted once on the training bank and frozen for all test queries.
    """
    def __init__(self, records):
        if len(records) < 2:
            raise ValueError('Need at least two bank examples')
        self.records = records
        self.questions = [row['question'] for row in records]
        self.solutions = [row['solution'] for row in records]
        self.steps, self.owners = [], []
        for index, solution in enumerate(self.solutions):
            parts = [line.strip() for line in solution.splitlines() if line.strip()]
            for part in parts:
                self.steps.append(part)
                self.owners.append(index)
        self.vectorizer = TfidfVectorizer(
            preprocessor=lexical_text, ngram_range=(1, 2),
            token_pattern=r'(?u)\b[a-zA-Z][a-zA-Z]+\b',
            max_features=12000, sublinear_tf=True,
        )
        self.vectorizer.fit(self.questions + self.steps)
        self.question_vectors = self.vectorizer.transform(self.questions)
        self.step_vectors = self.vectorizer.transform(self.steps)
        self.owners = np.asarray(self.owners)
        self.cache = {}

    def nearest(self, question, count=3):
        vector = self.vectorizer.transform([question])
        scores = (self.question_vectors @ vector.T).toarray().ravel()
        order = np.argsort(-scores, kind='stable')[:count]
        return [(int(i), float(scores[i])) for i in order if scores[i] > 0]

    def score(self, question, step, include_bank=True):
        if include_bank and question not in self.cache:
            nearest = self.nearest(question)
            owners = [index for index, _ in nearest]
            selected = np.flatnonzero(np.isin(self.owners, owners))
            self.cache[question] = selected
        selected = self.cache.get(question, []) if include_bank else []
        vector = self.vectorizer.transform([step])
        qvector = self.vectorizer.transform([question])
        relevance = float((vector @ qvector.T).toarray()[0, 0])
        preference = 0.0
        if len(selected):
            matches = (self.step_vectors[selected] @ vector.T).toarray().ravel()
            preference = float(matches.max())
        return relevance, preference


    def export(self, output):
        output = Path(output)
        with (output / 'bank_training_examples.jsonl').open('w', encoding='utf-8') as handle:
            for row in self.records:
                handle.write(json.dumps(row) + '\n')
        names = self.vectorizer.get_feature_names_out()
        weights = np.asarray(self.step_vectors.mean(axis=0)).ravel()
        table = pd.DataFrame({'phrase': names, 'mean_reference_tfidf': weights,
                              'idf': self.vectorizer.idf_})
        table.sort_values('mean_reference_tfidf', ascending=False).to_csv(
            output / 'word_bank_phrases.csv', index=False)
        # Save the exact fitted vocabulary/IDF, not an unsafe pickle.
        fitted = {'vocabulary': {k: int(v) for k, v in self.vectorizer.vocabulary_.items()},
                  'idf': self.vectorizer.idf_.tolist(),
                  'policy': 'Positive training-reference similarity; no negative preference labels.'}
        (output / 'word_bank_model.json').write_text(json.dumps(fitted, indent=2), encoding='utf-8')




In [ ]:
def prepare_word_bank(cache_dir, subsets, size=200):
    catalog = {key: read_snapshot(key, cache_dir)[0]
               for key in ('gsm8k_train', 'gsm8k_test', 'svamp')}
    excluded = {row['question_sha256'] for name in ('development', 'evaluation')
                for row in subsets[name]}
    excluded.update(row['question_sha256'] for row in catalog['gsm8k_test'] + catalog['svamp'])
    selected = fixed_sample(catalog['gsm8k_train'], size, 809, excluded)
    raw = [json.loads(line) for line in
           (Path(cache_dir) / 'gsm8k_train.jsonl').read_text(encoding='utf-8').splitlines()]
    records = []
    for row in selected:
        index = int(row['source_id'].rsplit(':', 1)[1])
        solution = raw[index]['answer'].split('####', 1)[0].strip()
        records.append({'source_id': row['source_id'], 'question': row['question'],
                        'question_sha256': row['question_sha256'], 'solution': solution})
    assert not {row['question_sha256'] for row in records} & excluded
    return ReferenceWordBank(records)




In [ ]:
class BudgetStop(BaseException):
    """Escape engine retries immediately when the shared budget is exhausted."""


class MeteredBackend:
    """Actual usage accounting plus a conservative preflight budget estimate.

    Ollama reports actual tokens after a call. The character estimate is not a
    tokenizer or a strict billable cap. Export any actual overshoot explicitly.
    """
    def __init__(self, backend, total_budget):
        self.backend, self.budget = backend, total_budget
        self.requests = []
        self.input_tokens = self.output_tokens = 0
        self.fallback = None

    @property
    def used(self):
        return self.input_tokens + self.output_tokens

    def __call__(self, prompt, schema, temperature, seed, max_tokens):
        estimated_input = math.ceil(len(prompt.encode('utf-8')) / 3) + 128
        remaining = self.budget - self.used - estimated_input
        if remaining < 64:
            raise BudgetStop('token_budget_preflight')
        limit = min(max_tokens, remaining)
        event = {'prompt': prompt, 'schema': schema, 'temperature': temperature,
                 'seed': seed, 'requested_output_limit': max_tokens,
                 'effective_output_limit': limit, 'estimated_input_tokens': estimated_input,
                 'usage_valid': False}
        self.requests.append(event)
        try:
            response = self.backend(prompt, schema, temperature, seed, limit)
        except Exception as exc:
            event['error'] = str(exc)
            raise
        event.update(response)
        if any(type(response.get(k)) is not int or response[k] < 0
               for k in ('prompt_tokens', 'completion_tokens')):
            event['error'] = 'missing_or_invalid_token_usage'
            raise RuntimeError('Backend must return actual nonnegative token usage')
        event['usage_valid'] = True
        self.input_tokens += response['prompt_tokens']
        self.output_tokens += response['completion_tokens']
        return response














In [ ]:
def search_prompt(question, steps, compact):
    if compact:
        # Deterministic compression: keep all equations and their labels; no summarizer call.
        history = '\n'.join(step['target'] + ': ' + step['equation'] for step in steps)
    else:
        history = json.dumps(steps)
    return ('Problem: ' + question + '\nPrior calculations: ' + history +
            '\nProduce ONE new arithmetic step using the schema. equation must contain only '
            'numbers, arithmetic operators, parentheses and one =. No variables or units in '
            'equation. target names the quantity calculated and its units for THIS step. '
            'Do not repeat calculations. Set done=false and answer=null until the whole '
            'question is solved. In the response computing the final quantity, set done=true '
            'and answer to that numeric result; do not leave the solved branch unfinished. '
            'Include expression (arithmetic only) and result (numeric value) as separate fields; '
            'they must agree when calculated locally. Give a nonempty step explanation. '
            'An empty final equation is allowed only if an earlier equation computed the answer.')




In [ ]:
def local_search(solver, question, bank, lexical=False, word_bank=False, compact=False):
    import heapq
    import itertools
    counter = itertools.count()
    queue = [(-1.0, next(counter), [])]
    best, expansions = [], 0
    seen_paths = set()
    rejected_terminals = {}
    stalled_branches = {}
    terminal_cache = {}
    depth_cutoff = False
    while queue and expansions < solver.cfg.max_expansions and solver.calls < solver.cfg.max_calls:
        _, _, steps = heapq.heappop(queue)
        if len(steps) >= solver.cfg.max_depth:
            depth_cutoff = True
            continue
        expansions += 1
        solver.events.append({'kind': 'local_expand', 'steps': steps, 'expansion': expansions})
        prompt = search_prompt(question, steps, compact)
        branch_key = json.dumps(steps, sort_keys=True)
        if branch_key in stalled_branches:
            prompt += ('\nThe previous expansion stalled: ' + stalled_branches[branch_key]['reason']
                       + '. Produce a new valid equation. If prior calculations already solve the '
                       'question, declare done=true with that answer and its requested target.')
        if branch_key in rejected_terminals:
            prompt += ('\nEarlier final answers on this branch failed validation: '
                       + json.dumps(rejected_terminals[branch_key])
                       + '. Recheck the requested quantity and give a corrected next step.')
        candidates = []
        for _ in range(solver.cfg.candidates):
            candidate = solver.request(prompt, STEP_SCHEMA, 0.7, 'local_generate', 384)
            if candidate is not None and candidate not in candidates:
                candidates.append(candidate)
        scored = []
        repeated_count = 0
        for candidate in candidates:
            key = equation_key(candidate['equation'])
            repeated = any(key == equation_key(step['equation']) for step in steps)
            if repeated and not candidate['done']:
                repeated_count += 1
                solver.events.append({'kind': 'local_repeated_step', 'candidate': candidate})
                continue
            relevance, preference = (bank.score(
                question, candidate['step'] + ' ' + candidate['target'], include_bank=word_bank)
                if lexical or word_bank else (0.0, 0.0))
            # Novel equation is a weak progress PROXY, not proof a subgoal was solved.
            valid, _ = equation_status(candidate['equation'])
            progress = float(valid == 'valid' and not repeated)
            history_text = ' '.join(step['step'].lower() for step in steps)
            words = set(re.findall(r'\b[a-z]+\b', candidate['step'].lower()))
            repetition = len(words & set(history_text.split())) / max(1, len(words))
            path = steps + [candidate]
            next_input = len(search_prompt(question, path, compact).encode('utf-8')) / 3 + 128
            generated = [e.get('completion_tokens', 0) for e in solver.events
                         if e.get('purpose') == 'local_generate' and 'completion_tokens' in e]
            expected_output = float(np.mean(generated)) if generated else 192.0
            estimated_cost = max(1.0, next_input + expected_output)
            utility = (0.2 + 0.5 * progress - 0.1 * repetition
                       + (0.15 * relevance if lexical else 0)
                       + (0.15 * preference if word_bank else 0))
            score = utility / estimated_cost
            solver.events.append({'kind': 'local_score', 'candidate': candidate,
                                  'question_tfidf': relevance, 'word_bank': preference,
                                  'progress_proxy': progress, 'repetition': repetition,
                                  'estimated_next_tokens': estimated_cost, 'score': score})
            scored.append((score, path))
        if not scored:
            reason = ('all candidates repeated earlier equations' if repeated_count else
                      solver.failure or 'no valid candidates generated')
            solver.events.append({'kind': 'branch_stalled', 'reason': reason, 'steps': steps})
            attempts = stalled_branches.get(branch_key, {}).get('attempts', 0)
            if attempts < solver.cfg.retries:
                stalled_branches[branch_key] = {'attempts': attempts + 1, 'reason': reason}
                heapq.heappush(queue, (-1.0, next(counter), steps))
        rejection = None
        for score, path in sorted(scored, key=lambda item: item[0], reverse=True):
            candidate = path[-1]
            if candidate['done']:
                values = [equation_status(step['equation'])[1] for step in path]
                grounded = any(v is not None and math.isclose(
                    v, candidate['answer'], abs_tol=1e-4, rel_tol=0) for v in values)
                # All four local-search ablations use this same counted review.
                terminal_key = (tuple(equation_key(step['equation']) for step in path),
                                candidate['answer'], candidate['target'].strip().lower())
                if terminal_key not in terminal_cache:
                    terminal_cache[terminal_key] = (grounded and
                                                    solver.terminal_review(question, path[:-1], candidate))
                else:
                    solver.events.append({'kind': 'terminal_cache_hit', 'candidate': candidate})
                if terminal_cache[terminal_key]:
                    return candidate['answer'], path, 'done'
                rejection = {'candidate': candidate, 'grounded': grounded}
                solver.events.append({'kind': 'local_terminal_rejected', **rejection})
            else:
                signature = tuple(equation_key(step['equation']) for step in path)
                if signature not in seen_paths:
                    seen_paths.add(signature)
                    heapq.heappush(queue, (-score, next(counter), path))
                    if len(path) > len(best):
                        best = path
        # At most cfg.retries recovery expansions per branch. No reference-answer feedback.
        if rejection is not None:
            previous = rejected_terminals.setdefault(branch_key, [])
            if len(previous) < solver.cfg.retries:
                previous.append(rejection)
                heapq.heappush(queue, (-max(score for score, _ in scored), next(counter), steps))
        queue = heapq.nsmallest(solver.cfg.beam, queue)
        heapq.heapify(queue)
    status = ('call_limit' if solver.calls >= solver.cfg.max_calls else
              'expansion_limit' if expansions >= solver.cfg.max_expansions else
              'depth_limit' if depth_cutoff else 'exhausted')
    return None, best, status






In [ ]:
def failure_diagnostics(events, status, last_error):
    counts = Counter()
    last_reason = last_error
    for event in events:
        reason = event.get('error')
        if event.get('kind') == 'branch_stalled':
            reason = event['reason']
        elif event.get('kind') == 'local_repeated_step':
            reason = 'repeated_equation'
        elif event.get('kind') in ('local_terminal_rejected', 'rejected'):
            reason = event.get('reason', 'terminal_review_or_grounding_rejected')
        elif event.get('kind') == 'terminal_decision' and not event.get('accepted'):
            review = event.get('review')
            reason = 'terminal_review: ' + str(review.get('reason') if review else 'no valid review')
        if reason:
            counts[str(reason)] += 1
            last_reason = str(reason)
    return {'failure_reason': last_reason or status, 'diagnostic_counts': dict(counts)}




In [ ]:
def run_method(backend, question, method, bank, seed, token_budget=16000, config=None):
    if method not in ALL_METHODS:
        raise ValueError('Unknown method: ' + method)
    if type(token_budget) is not int or token_budget < 1:
        raise ValueError('Token budget must be a positive integer')
    if type(seed) is not int or not 0 <= seed < 2**31:
        raise ValueError('Seed must be an integer from 0 to 2**31 - 1')
    started = time.perf_counter()
    cfg = replace(config or Config(max_calls=30, max_expansions=16, retries=1,
                                   audit_tos=True, review_appeal=True), seed=seed)
    meter = MeteredBackend(backend, token_budget)
    solver = Solver(meter, cfg)
    answer, steps, status = None, [], 'missing'
    extra = {}
    # Reset query-derived cache for comparable per-solver timing; fitted training bank stays frozen.
    bank.cache.clear()
    try:
        if method in METHODS:
            result = solver.solve(question, method)
            answer, steps, status = result['ans'], result['steps'], result['status']
            extra = {k: result[k] for k in ('baseline_version', 'search_model') if k in result}
        else:
            answer, steps, status = local_search(
                solver, question, bank, lexical=method != NEW_METHODS[0],
                word_bank=method in NEW_METHODS[2:4], compact=method == NEW_METHODS[3])
        if answer is not None and status in ('missing', 'exhausted'):
            status = 'done'
        elif answer is None and status == 'missing':
            status = solver.failure or 'missing_final_answer'
    except BudgetStop:
        answer = meter.fallback
        status = 'budget_fallback' if answer is not None else 'token_budget'
    except Exception as exc:
        status = 'runtime_error'
        solver.events.append({'kind': 'runtime_error', 'error': repr(exc)})
    diagnostics = failure_diagnostics(solver.events, status, solver.failure) if answer is None else {}
    return dict(method=method, ans=answer, status=status, steps=steps,
                calls=len(meter.requests), tokens=meter.used, input_tokens=meter.input_tokens,
                output_tokens=meter.output_tokens, sec=time.perf_counter() - started,
                trace=solver.events, requests=meter.requests, seed=seed, config=vars(cfg),
                token_budget=token_budget, budget_overshoot=max(0, meter.used - token_budget),
                usage_complete=all(r.get('usage_valid', False) for r in meter.requests),
                engine_version=EXPERIMENT_VERSION, last_error=solver.failure, **extra, **diagnostics)




In [ ]:
def validate_prices(input_price, output_price):
    if (input_price is None) != (output_price is None):
        raise ValueError('Supply both input and output prices or neither')
    if input_price is not None and any(not numeric(v) or v < 0 for v in (input_price, output_price)):
        raise ValueError('Prices must be finite nonnegative numbers')


def experiment_report(rows, output, input_price=None, output_price=None):
    import matplotlib.pyplot as plt
    validate_prices(input_price, output_price)
    if not rows:
        raise ValueError('No completed solver records to report')
    output = Path(output)
    df = pd.DataFrame([{k: v for k, v in row.items() if k not in
                        ('trace', 'requests', 'steps', 'search_model', 'config')} for row in rows])
    summaries = []
    for (dataset, method), group in df.groupby(['dataset', 'method'], sort=False):
        correct = int(group['correct'].sum())
        complete = bool(group['usage_complete'].all())
        summaries.append(dict(
            dataset=dataset, method=method, runs=len(group), correct=correct,
            missing=int(group['ans'].isna().sum()),
            wrong=int(((~group['correct']) & group['ans'].notna()).sum()),
            accuracy_pct=100 * correct / len(group), mean_calls=group['calls'].mean(),
            mean_input_tokens=group['input_tokens'].mean() if complete else np.nan,
            mean_output_tokens=group['output_tokens'].mean() if complete else np.nan,
            mean_tokens=group['tokens'].mean() if complete else np.nan,
            mean_observed_tokens=group['tokens'].mean(),
            tokens_per_correct=group['tokens'].sum() / correct if correct and complete else np.nan,
            mean_seconds=group['sec'].mean(), budget_overshoot_runs=int((group['budget_overshoot'] > 0).sum()),
            usage_incomplete_runs=int((~group['usage_complete']).sum()),
        ))
    table = pd.DataFrame(summaries)
    for dataset in table['dataset'].unique():
        mask = table['dataset'] == dataset
        reference = table[mask & (table['method'] == METHODS[0])]
        if not reference.empty and reference.iloc[0]['mean_tokens'] > 0:
            table.loc[mask, 'token_savings_vs_cot_pct'] = (
                100 * (1 - table.loc[mask, 'mean_tokens'] / reference.iloc[0]['mean_tokens']))
    if input_price is not None:
        table['hypothetical_usd_per_problem'] = (
            table['mean_input_tokens'] * input_price + table['mean_output_tokens'] * output_price) / 1e6
    table.to_csv(output / 'comparison_summary.csv', index=False)
    df.to_csv(output / 'per_problem_metrics.csv', index=False)
    columns = ['dataset', 'source_id', 'method', 'seed', 'status', 'failure_reason',
               'diagnostic_counts', 'calls', 'input_tokens', 'output_tokens']
    df.reindex(columns=columns).loc[df['ans'].isna()].to_csv(output / 'missing_diagnostics.csv', index=False)
    df['complete_tokens'] = df['tokens'].where(df['usage_complete'])
    df.groupby(['dataset', 'method', 'seed']).agg(
        accuracy=('correct', 'mean'), tokens=('complete_tokens', lambda s: s.mean(skipna=False)),
        calls=('calls', 'mean')
    ).to_csv(output / 'per_seed_metrics.csv')
    df.groupby(['dataset', 'method', 'status']).size().to_csv(output / 'termination_counts.csv')
    for dataset in table['dataset'].unique():
        sub = table[table['dataset'] == dataset]
        fig, ax = plt.subplots(figsize=(12, 7))
        for _, row in sub.iterrows():
            if not math.isfinite(row['mean_tokens']):
                continue
            ax.scatter(row['mean_tokens'], row['accuracy_pct'])
            ax.annotate(row['method'], (row['mean_tokens'], row['accuracy_pct']),
                        fontsize=7, xytext=(3, 4), textcoords='offset points')
        ax.set(xlabel='Mean total input + output tokens per problem',
               ylabel='Accuracy (%) — missing answers count as failures', title=dataset, ylim=(-2, 105))
        ax.grid(alpha=0.2)
        fig.tight_layout()
        fig.savefig(output / (dataset + '_accuracy_vs_tokens.png'), dpi=160)
        plt.close(fig)
    return table




In [ ]:
def run_word_bank_suite(backend, cases, bank, output, seeds, methods, token_budget,
                       provenance, profile, config=None, input_price=None, output_price=None):
    validate_prices(input_price, output_price)
    if not cases or not seeds or len(set(seeds)) != len(seeds):
        raise ValueError('Need cases and distinct seeds')
    if any(type(seed) is not int or not 0 <= seed < 2**31 for seed in seeds):
        raise ValueError('Seeds must be integers from 0 to 2**31 - 1')
    if not methods or len(set(methods)) != len(methods) or any(m not in ALL_METHODS for m in methods):
        raise ValueError('Need distinct known methods')
    if type(token_budget) is not int or token_budget < 256:
        raise ValueError('Token budget must be an integer of at least 256')
    for case in cases:
        if (not isinstance(case.get('question'), str) or not case['question'].strip()
                or not numeric(case.get('expected')) or not isinstance(case.get('source_id'), str)
                or not isinstance(case.get('dataset'), str)
                or not re.fullmatch(r'[a-zA-Z0-9_-]+', case['dataset'])):
            raise ValueError('Each case needs a question, numeric reference, source ID and dataset name')
    splits = {case.get('evaluation_split', profile) for case in cases}
    if len(splits) != 1 or ('evaluation' in splits) != (profile == 'evaluation'):
        raise ValueError('Do not mix development and evaluation profiles')
    if len({case['source_id'] for case in cases}) != len(cases):
        raise ValueError('Duplicate evaluation source IDs')
    bank_hashes = {row['question_sha256'] for row in bank.records}
    if any(fingerprint(case['question']) in bank_hashes for case in cases):
        raise ValueError('Training-bank leakage into benchmark')
    output = Path(output)
    output.mkdir(parents=True, exist_ok=False)
    bank.export(output)
    cfg = config or Config(max_calls=30, max_expansions=16, retries=1)
    manifest = dict(version=EXPERIMENT_VERSION, model='qwen2.5:7b', profile=profile,
                    methods=list(methods), seeds=list(seeds), token_budget=token_budget,
                    config=vars(cfg), provenance=provenance,
                    bank_source_ids=[row['source_id'] for row in bank.records],
                    bank_selection_seed=809, bank_policy='Positive gold TRAINING reference steps only',
                    baseline_policy='Existing methods with shared experiment limits; not old run settings',
                    prices_per_million=dict(input=input_price, output=output_price),
                    budget_policy='Estimated preflight; actual usage and overshoot recorded',
                    planned_runs=len(cases) * len(seeds) * len(methods))
    (output / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    with (output / 'selected_problems.jsonl').open('w', encoding='utf-8') as handle:
        for case in cases:
            handle.write(json.dumps(case) + '\n')
    rows = []
    with (output / 'full_results.jsonl').open('w', encoding='utf-8') as handle:
        for seed in seeds:
            for case in cases:
                # Rotate method order deterministically to reduce fixed-order timing bias.
                offset = int(hashlib.sha256(f'{seed}:{case["source_id"]}'.encode()).hexdigest(), 16)
                offset %= len(methods)
                order = list(methods[offset:]) + list(methods[:offset])
                for method in order:
                    result = run_method(backend, case['question'], method, bank, seed, token_budget, cfg)
                    # Reference answers are accessed ONLY after the solver returns.
                    result.update({key: case[key] for key in
                                   ('dataset', 'source_id', 'question', 'expected')})
                    answer = result['ans']
                    result['correct'] = answer is not None and math.isclose(
                        answer, case['expected'], abs_tol=1e-4, rel_tol=0)
                    result['outcome'] = ('missing' if answer is None else
                                         'correct' if result['correct'] else 'wrong')
                    result['experiment_id'] = output.name
                    result['profile'] = profile
                    handle.write(json.dumps(result) + '\n')
                    handle.flush()
                    rows.append(result)
                    print(seed, case['source_id'], method, result['outcome'],
                          result['tokens'], 'tokens', result['status'],
                          ('| ' + result.get('failure_reason', '')[:240]) if answer is None else '', flush=True)
    table = experiment_report(rows, output, input_price, output_price)
    return rows, table


In [ ]:
EMBEDDED_SOURCE_HASHES = {
    "benchmark_engine": "3f5c1172e83e5760978d37c94cc6b59dd072738b49f2cf8c0e31e31605e75a87",
    "tos_search": "b06fb552977b6d49255157d9667d189b9e677cc2d74f5e747b1f098d751d1de7",
    "benchmark_data": "b0662d2ba63192d370cf4723589bba21b18cd87d8aef29b8104018c393e122cf",
    "word_bank_experiment": "86454c15930a68240d06f94a9e5037501a2e233cf0c44dc94b0d3b717c2d5694"
}


## 2. Experiment settings


In [ ]:
# Edit this cell before a run. Defaults keep the first diagnostic manageable.
PROFILE = 'pilot'             # 'pilot', 'development', or 'evaluation'
LIMIT_PER_DATASET = 5          # None uses the full selected profile
SEEDS = [11]                  # For a final run, consider [11, 23, 37]
METHODS_TO_RUN = list(ALL_METHODS)
BANK_TRAINING_SIZE = 200
TOKEN_BUDGET_PER_PROBLEM = 16000
SEARCH_CONFIG = Config(max_calls=30, max_expansions=16, retries=1,
                       max_depth=10, beam=3, candidates=2,
                       audit_tos=True, review_appeal=True)

# To focus on the four local-search ablations:
# METHODS_TO_RUN = [METHODS[0], *NEW_METHODS]

# Optional hypothetical API prices in USD per million tokens. Local Ollama has no API invoice.
INPUT_USD_PER_MILLION = None
OUTPUT_USD_PER_MILLION = None

# Full evaluation: PROFILE='evaluation', LIMIT_PER_DATASET=None, SEEDS=[11, 23, 37].
# With all 9 methods: 150 questions x 3 seeds x 9 methods = 4050 solver runs.
# Freeze methods, weights and budgets BEFORE inspecting evaluation outcomes.


## 3. Prepare disjoint datasets and training phrase bank


In [ ]:
CACHE_DIR = WORK_DIR / 'word_bank_data_cache'
subsets, data_provenance = prepare_benchmarks(cache_dir=CACHE_DIR)
if PROFILE not in subsets:
    raise ValueError('Choose pilot, development or evaluation')
if LIMIT_PER_DATASET is not None and (type(LIMIT_PER_DATASET) is not int or LIMIT_PER_DATASET < 1):
    raise ValueError('LIMIT_PER_DATASET must be a positive integer or None')
if type(BANK_TRAINING_SIZE) is not int or BANK_TRAINING_SIZE < 2:
    raise ValueError('BANK_TRAINING_SIZE must be an integer of at least two')
selected_problems = []
counts = Counter()
for problem in subsets[PROFILE]:
    dataset = problem['dataset']
    if LIMIT_PER_DATASET is None or counts[dataset] < LIMIT_PER_DATASET:
        selected_problems.append(problem)
        counts[dataset] += 1

bank_started = time.perf_counter()
word_bank = prepare_word_bank(CACHE_DIR, subsets, BANK_TRAINING_SIZE)
bank_build_seconds = time.perf_counter() - bank_started
print('Selected:', dict(counts), '| training-bank examples:', len(word_bank.records))
print('Planned solver runs:', len(selected_problems) * len(SEEDS) * len(METHODS_TO_RUN))
print('Bank construction seconds:', round(bank_build_seconds, 3), '| LLM calls: 0')
print('Methods:')
for index, method in enumerate(METHODS_TO_RUN, 1):
    print(index, method)


## 4. Run all selected approaches


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
import importlib.metadata
from IPython.display import display

OUTPUT = WORK_DIR / ('word_bank_results_' + datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
                     + '_' + uuid4().hex[:6])
runtime_provenance = dict(data_provenance)
runtime_provenance.update(
    source_hashes=EMBEDDED_SOURCE_HASHES, bank_build_seconds=bank_build_seconds,
    package_versions={name: importlib.metadata.version(name) for name in
                      ('ollama', 'scikit-learn', 'numpy', 'pandas', 'matplotlib')},
)
# Capture installed model identity so mutable model tags do not obscure reproducibility.
import ollama
runtime_provenance['ollama_models'] = ollama.list().model_dump(mode='json')
print('Results will be written after each solver run to:', OUTPUT)
all_results, summary = run_word_bank_suite(
    ollama_backend, selected_problems, word_bank, OUTPUT, SEEDS, METHODS_TO_RUN,
    TOKEN_BUDGET_PER_PROBLEM, runtime_provenance, PROFILE, SEARCH_CONFIG,
    INPUT_USD_PER_MILLION, OUTPUT_USD_PER_MILLION,
)
display(summary)


## 5. Compare differences


In [ ]:
# Isolate one change at a time. Positive accuracy delta is better;
# negative token delta is cheaper. No significance claim from a small pilot.
from IPython.display import Image, display
pairs = [
    (NEW_METHODS[0], NEW_METHODS[1], 'Add question TF-IDF'),
    (NEW_METHODS[1], NEW_METHODS[2], 'Add reference word bank'),
    (NEW_METHODS[2], NEW_METHODS[3], 'Compact the reasoning state'),
    (METHODS[4], METHODS[3], 'Add predicted next-call token penalty'),
    (METHODS[2], METHODS[3], 'Replace LLM step judge with local ranking'),
]
differences = []
for dataset in summary['dataset'].unique():
    table = summary[summary['dataset'] == dataset].set_index('method')
    for control, treatment, change in pairs:
        if control in table.index and treatment in table.index:
            a, b = table.loc[control], table.loc[treatment]
            differences.append({
                'dataset': dataset, 'change': change,
                'control': control, 'treatment': treatment,
                'accuracy_delta_percentage_points': b['accuracy_pct'] - a['accuracy_pct'],
                'mean_token_delta': b['mean_tokens'] - a['mean_tokens'],
                'mean_call_delta': b['mean_calls'] - a['mean_calls'],
            })
difference_table = pd.DataFrame(differences, columns=[
    'dataset', 'change', 'control', 'treatment', 'accuracy_delta_percentage_points',
    'mean_token_delta', 'mean_call_delta',
])
difference_table.to_csv(OUTPUT / 'ablation_differences.csv', index=False)
display(difference_table)
for chart in sorted(OUTPUT.glob('*_accuracy_vs_tokens.png')):
    display(Image(filename=str(chart)))


## Reading the results

Check accuracy, missing answers and wrong answers together. Compare total **input + output**
tokens; fewer calls do not necessarily mean lower cost. `tokens_per_correct` divides all
spent tokens, including failures, by the number of correct answers. Zero correct answers
produce an undefined value. Each dataset is reported separately.

Seeds repeat questions and are not independent new test questions. Use a larger held-out
run before drawing conclusions; uncertainty analysis should resample question IDs, keeping
methods paired and seeds grouped. This notebook does not claim statistical significance.

A word-bank improvement means the same search configuration solves as many or more
questions for fewer tokens, or improves accuracy at a comparable budget. More positive
similarity scores alone are not an improvement. Inspect full traces for counterexamples.

Artifacts: `comparison_summary.csv`, `ablation_differences.csv`, `per_problem_metrics.csv`,
`per_seed_metrics.csv`, `termination_counts.csv`, charts, `full_results.jsonl`, manifest,
`missing_diagnostics.csv` (specific failure reasons and rejection counts),
selected questions, training examples, fitted vocabulary/IDF, and `word_bank_phrases.csv`.
Runtime includes local scoring; bank build time is recorded separately. Training references
have no LLM construction cost here, but curating real preference labels would have a cost.
Monetary values are hypothetical only when both prices were supplied.


## 6. Download results


In [ ]:
# Also works after an interrupted benchmark, once OUTPUT was created.
import shutil
from IPython.display import FileLink, display

if not OUTPUT.exists():
    raise RuntimeError('No results directory exists yet.')
archive = shutil.make_archive(str(OUTPUT), 'zip', str(OUTPUT))
print('Download this archive from the link or Kaggle Output panel:')
display(FileLink(Path(archive).name))
